# Project: A neural net in pure NumPy 🧠
No PyTorch, no autograd: write the forward pass and **backpropagation by hand** for a 2-layer network, verify your gradients numerically (the check every DL researcher has done once), and train it to untangle the two-moons dataset.

Topic: [[Backpropagation]], [[Neural Networks]] · guide note: [[Project - Neural Net in Pure NumPy]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_moons
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
X, y = make_moons(600, noise=0.2, random_state=0)
rng = np.random.default_rng(0); H = 32
params = [rng.normal(0, np.sqrt(2 / 2), (2, H)), np.zeros(H), rng.normal(0, np.sqrt(2 / H), (H, 2)), np.zeros(2)]   # He init

## 1. Forward pass\n$z_1 = XW_1+b_1$, $h=\text{ReLU}(z_1)$, logits $=hW_2+b_2$, then a **numerically stable softmax** (subtract the row max). Return `(probabilities, cache)` with `cache = (X, z1, h)`.

In [ ]:
def forward(X, W1, b1, W2, b2):
    # TODO 1: linear → ReLU → linear → stable softmax
    raise NotImplementedError  # TODO 1

## 2. Backward pass\nFor mean cross-entropy, $\frac{\partial L}{\partial \text{logits}} = (p - \text{onehot}(y))/n$. Then chain backwards: $dW_2=h^\top d\text{logits}$, $dh = d\text{logits}\,W_2^\top$, $dz_1 = dh\odot[z_1>0]$, $dW_1 = X^\top dz_1$. Return `(dW1, db1, dW2, db2)`.

In [ ]:
def backward(p, y, cache, W2):
    # TODO 2: gradients of the mean cross-entropy w.r.t. all parameters
    raise NotImplementedError  # TODO 2

In [ ]:
def loss(params, X, y):
    p, _ = forward(X, *params); return -np.mean(np.log(p[np.arange(len(y)), y] + 1e-12))

def _t_grad():
    r = np.random.default_rng(1); P = [r.normal(0, 1, (2, 5)), r.normal(0, 1, 5), r.normal(0, 1, (5, 2)), r.normal(0, 1, 2)]
    Xs, ys = r.normal(0, 1, (7, 2)), r.integers(0, 2, 7)
    p, cache = forward(Xs, *P); grads = backward(p, ys, cache, P[2])
    worst = 0
    for k in range(4):                                    # central finite differences on every parameter
        num = np.zeros_like(P[k])
        for idx in np.ndindex(P[k].shape):
            old = P[k][idx]; P[k][idx] = old + 1e-5; lp = loss(P, Xs, ys); P[k][idx] = old - 1e-5; lm = loss(P, Xs, ys); P[k][idx] = old
            num[idx] = (lp - lm) / 2e-5
        worst = max(worst, np.max(np.abs(num - grads[k])) / (np.max(np.abs(num)) + 1e-12))
    return np.allclose(p.sum(1), 1) and worst < 1e-5
check("gradient check", _t_grad, "Analytic and numerical gradients must agree to ~1e-5 (relative); probabilities must sum to 1.")
if ready("gradient check"):
    hist = []
    for step in range(3000):
        p, cache = forward(X, *params); grads = backward(p, y, cache, params[2])
        params = [w - 0.5 * g for w, g in zip(params, grads)]; hist.append(loss(params, X, y))
    acc = np.mean(forward(X, *params)[0].argmax(1) == y)
    xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1, 1.5, 200))
    zz = forward(np.c_[xx.ravel(), yy.ravel()], *params)[0][:, 1].reshape(xx.shape)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6)); ax[0].plot(hist); ax[0].set_title("training loss")
    ax[1].contourf(xx, yy, zz, 20, cmap="RdBu", alpha=.6); ax[1].scatter(*X.T, c=y, cmap="RdBu", s=6); ax[1].set_title(f"decision boundary, accuracy {acc:.1%}"); plt.show()
    check("learns the moons", lambda: acc > 0.95, "A 32-unit ReLU net should reach > 95 % training accuracy on the moons.")

<details><summary>▶ Solution</summary>

```python
def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    h = np.maximum(0, z1)
    logits = h @ W2 + b2
    logits = logits - logits.max(1, keepdims=True)
    p = np.exp(logits) / np.exp(logits).sum(1, keepdims=True)
    return p, (X, z1, h)
```

```python
def backward(p, y, cache, W2):
    X, z1, h = cache
    n = len(y)
    dlogits = p.copy(); dlogits[np.arange(n), y] -= 1; dlogits /= n
    dW2 = h.T @ dlogits; db2 = dlogits.sum(0)
    dh = dlogits @ W2.T
    dz1 = dh * (z1 > 0)
    dW1 = X.T @ dz1; db1 = dz1.sum(0)
    return dW1, db1, dW2, db2
```
</details>

## Stretch goals
- Replace ReLU with tanh (change forward *and* backward) and compare training speed.
- Add mini-batches and momentum ([[Optimizers]]).
- Check your network against PyTorch: same initial weights → same gradients?